# Method C-DPO - PL Log-Ratio plus DPO Core — OLMo-2-0425-1B-SFT Full FT / Kaggle P100

## Overview
This notebook is copied from the original VDPO training notebook and adapted for the new combined Intel-Orca segmented preference dataset.

**Active variant:** `C-DPO`

**Objective:** `Delta = beta * (h_chosen - h_rejected) + U_theta_omega(chosen) - U_theta_omega(rejected)`

Standard DPO response-level core augmented with Method A's score-gap-weighted, reference-baselined PL structural utility.

### Dataset Format
The new dataset stores one preference pair per row:

```json
{
  "index": 0,
  "prompt": "...",
  "positive_response": "...",
  "positive_segments": [{"text": "...", "value_score": 0.96, "rank": 1}],
  "negative_response": "...",
  "negative_segments": [{"text": "...", "value_score": 0.14, "rank": 6}]
}
```

Rows must have both segment lists to train these methods. Segment `rank` is used for PL order; `value_score` is used as the importance weight when it is in `[0, 1]`. If a row has out-of-range scores, weights are derived from `rank` for that response.

### OLMo 2 full-model setup
This copy performs **full-parameter optimization** of `allenai/OLMo-2-0425-1B-SFT` (no LoRA and no weight quantization). The 16 GB P100 defaults use FP16 policy/reference weights, gradient checkpointing, an 8-bit AdamW optimizer state, micro-batch 1, and a 512-token alignment window. The optimizer state is compressed, but every model parameter remains trainable and checkpoints contain the full model.


## Cell 1 — Install Dependencies

In [ ]:
# Install required packages (run once, then restart the Kaggle kernel)
# OLMo is distributed as a PyTorch .bin checkpoint. Transformers requires
# torch>=2.6 to load it safely (CVE-2025-32434). Official CUDA 11.8 wheels
# retain sm_60 support for Tesla P100; all three PyTorch packages must match.
!pip install -q -U \
    torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu118

!pip install -q -U \
    transformers==5.6.2 \
    "accelerate>=1.12.0" \
    bitsandbytes==0.48.1 \
    "huggingface_hub>=0.36.0" \
    "safetensors>=0.4.5" \
    sentencepiece "protobuf<6" tqdm matplotlib


## Cell 2 — Imports & Reproducibility

In [ ]:
import json
import os

# Must be set before the first CUDA allocation. Variable-length DPO batches
# otherwise leave non-coalescible reserved blocks on a 16 GB P100.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import random
import math
from pathlib import Path
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import bitsandbytes as bnb
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login

    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("Huggingface")
    HF_TOKEN = secret_value_0
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Loaded Hugging Face token from Kaggle secret: Huggingface")
except Exception as exc:
    print(
        "Kaggle Hugging Face secret was not loaded; continuing with public or "
        f"locally authenticated Hugging Face access. ({type(exc).__name__}: {exc})"
    )

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


## Cell 3 — Dataset Configuration
Point the notebook at the merged JSONL exported from this repo. By default we use the
`all` export so every indexed positive/negative pair is available for training.


In [ ]:
METHOD_VARIANT = "C-DPO"
METHOD_DESCRIPTION = "Standard DPO response-level core augmented with Method A's score-gap-weighted, reference-baselined PL structural utility."
DATA_PATH = None  # Optional override for a custom JSONL path

DEFAULT_DATASET_RELATIVE_PATHS = [
    "data/new dataset/intel_orca_positive_negative_combined_final.jsonl",
    "data/new dataset/intel_orca_positive_negative_combined.jsonl",
    "/kaggle/input/intel-orca-positive-negative-combined/intel_orca_positive_negative_combined_final.jsonl",
    "/kaggle/input/intel-orca-positive-negative-combined/intel_orca_positive_negative_combined.jsonl",
    "/kaggle/input/datasets/ahmadsubhaniiqbal/intel-orca-dpo-pairs-segmented/intel_orca_positive_negative_combined_final.jsonl",
    "/kaggle/input/datasets/ahmadsubhaniiqbal/intel-orca-dpo-pairs-segmented/intel_orca_positive_negative_combined.jsonl",
]


def candidate_repo_roots(start: Path) -> List[Path]:
    return [start, *start.parents]


def resolve_jsonl_path(explicit_path: Optional[str] = None) -> Path:
    search_paths: List[Path] = []
    if explicit_path:
        explicit = Path(explicit_path).expanduser()
        search_paths.append(explicit)
        if not explicit.is_absolute():
            search_paths.extend((root / explicit) for root in candidate_repo_roots(Path.cwd()))
    else:
        for raw_path in DEFAULT_DATASET_RELATIVE_PATHS:
            candidate = Path(raw_path).expanduser()
            search_paths.append(candidate)
            if not candidate.is_absolute():
                search_paths.extend((root / candidate) for root in candidate_repo_roots(Path.cwd()))

    seen = set()
    for candidate in search_paths:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if resolved.exists():
            return resolved

    raise FileNotFoundError(
        "Could not find the new combined dataset JSONL. Set DATA_PATH to the file you want to train on."
    )


def inspect_dataset_schema(path: Path) -> Dict[str, int]:
    stats = {
        "rows": 0,
        "usable_combined_rows": 0,
        "missing_positive_segments": 0,
        "missing_negative_segments": 0,
        "empty_positive_segments": 0,
        "empty_negative_segments": 0,
        "rows_with_out_of_range_scores": 0,
    }
    example_keys = None
    segment_keys = None

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            stats["rows"] += 1
            if example_keys is None:
                example_keys = sorted(rec.keys())

            pos_segments = rec.get("positive_segments")
            neg_segments = rec.get("negative_segments")
            if pos_segments is None:
                stats["missing_positive_segments"] += 1
            elif not pos_segments:
                stats["empty_positive_segments"] += 1
            if neg_segments is None:
                stats["missing_negative_segments"] += 1
            elif not neg_segments:
                stats["empty_negative_segments"] += 1

            if pos_segments and neg_segments:
                stats["usable_combined_rows"] += 1

            row_out_of_range = False
            for segments in (pos_segments or [], neg_segments or []):
                for seg in segments:
                    if segment_keys is None:
                        segment_keys = sorted(seg.keys())
                    try:
                        score = float(seg.get("value_score", 0.0))
                    except (TypeError, ValueError):
                        score = 0.0
                    if score < 0.0 or score > 1.0:
                        row_out_of_range = True
            if row_out_of_range:
                stats["rows_with_out_of_range_scores"] += 1

    print(f"Method variant: {METHOD_VARIANT}")
    print(f"Method summary: {METHOD_DESCRIPTION}")
    print(f"Using dataset JSONL: {path}")
    print(f"Dataset rows found: {stats['rows']}")
    print(f"Rows with both segmented responses: {stats['usable_combined_rows']}")
    print(
        "Rows skipped if training requires both segment lists: "
        f"positive_missing={stats['missing_positive_segments']}, "
        f"positive_empty={stats['empty_positive_segments']}, "
        f"negative_missing={stats['missing_negative_segments']}, "
        f"negative_empty={stats['empty_negative_segments']}"
    )
    print(f"Rows with out-of-range segment scores: {stats['rows_with_out_of_range_scores']}")
    print(f"Top-level keys: {example_keys}")
    print(f"Segment keys: {segment_keys}")
    return stats


DATA_PATH = resolve_jsonl_path(DATA_PATH)
dataset_schema_stats = inspect_dataset_schema(DATA_PATH)


## Cell 4 - Tokenizer and Segment Metadata Builder

The new combined dataset stores one preference pair per row with `positive_response`, `negative_response`, `positive_segments`, and `negative_segments`. Segment records carry `text`, `value_score`, and `rank`. The tokenizer helper aligns each response token back to its segment id, score, and rank so the loss can compute both response-level and PL-structural terms.


In [ ]:
MODEL_NAME = "allenai/OLMo-2-0425-1B-SFT"

# P100 has fast FP16 but no native BF16. This capability check also makes the
# notebook choose BF16 automatically when moved to a newer supporting GPU.
if DEVICE == "cuda" and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
elif DEVICE == "cuda":
    MODEL_DTYPE = torch.float16
else:
    MODEL_DTYPE = torch.float32

if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    gpu_gib = gpu.total_memory / 1024**3
    print(f"GPU: {gpu.name} ({gpu_gib:.1f} GiB)")
    if gpu_gib < 14.5:
        print("WARNING: these defaults target a 16 GB GPU.")

print(f"Loading tokenizer from {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=False,
    token=HF_TOKEN,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocab size: {len(tokenizer)}  |  Pad token: '{tokenizer.pad_token}'")
print(f"Training dtype: {MODEL_DTYPE} (P100 should report torch.float16)")


In [ ]:
def _coerce_float(value, default: float = 0.0) -> float:
    try:
        out = float(value)
    except (TypeError, ValueError):
        return default
    return out if math.isfinite(out) else default


def _coerce_rank(value, default: Optional[int] = None) -> Optional[int]:
    try:
        rank = int(value)
    except (TypeError, ValueError):
        return default
    return rank if rank > 0 else default


def normalize_segment_records(segments: List[Dict]) -> List[Dict]:
    """
    Return clean segment records with stable score and rank fields.

    Normal rows keep their original value_score in [0, 1]. A small number of
    rows in the new dataset have score-like values greater than 1. For those
    rows, scores are derived from the explicit rank field so rank 1 remains the
    most important segment.
    """
    cleaned: List[Dict] = []
    for position, seg in enumerate(segments or []):
        if not isinstance(seg, dict):
            continue
        text = str(seg.get("text", ""))
        if text == "":
            continue
        score = _coerce_float(seg.get("value_score"), 0.0)
        rank = _coerce_rank(seg.get("rank"), None)
        cleaned.append({
            "text": text,
            "value_score": score,
            "rank": rank,
            "_source_position": position,
        })

    if not cleaned:
        return []

    ranks_available = all(seg["rank"] is not None for seg in cleaned)
    if not ranks_available:
        order = sorted(
            range(len(cleaned)),
            key=lambda i: (-cleaned[i]["value_score"], cleaned[i]["_source_position"]),
        )
        for rank_position, seg_idx in enumerate(order, start=1):
            cleaned[seg_idx]["rank"] = rank_position

    score_out_of_range = any(
        seg["value_score"] < 0.0 or seg["value_score"] > 1.0
        for seg in cleaned
    )
    if score_out_of_range:
        order = sorted(
            range(len(cleaned)),
            key=lambda i: (cleaned[i]["rank"], cleaned[i]["_source_position"]),
        )
        m = len(order)
        rank_derived_scores = {}
        for order_position, seg_idx in enumerate(order):
            rank_derived_scores[seg_idx] = float(m - order_position) / float(m)
        for seg_idx, seg in enumerate(cleaned):
            seg["value_score"] = rank_derived_scores[seg_idx]
            seg["score_source"] = "rank_derived"
    else:
        for seg in cleaned:
            seg["value_score"] = min(max(float(seg["value_score"]), 0.0), 1.0)
            seg["score_source"] = "value_score"

    for seg in cleaned:
        seg["rank"] = int(seg["rank"])

    return cleaned


def build_token_segment_metadata(
    response_text: str,
    segments: List[Dict],
    tokenizer,
    default_score: float = 0.0,
    default_segment_id: int = -1,
    default_rank: int = -1,
) -> Tuple[List[int], List[float], List[int], List[int]]:
    """Tokenize `response_text` and align each token to segment score/id/rank."""
    normalized_segments = normalize_segment_records(segments)
    enc = tokenizer(
        response_text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    token_ids: List[int] = enc["input_ids"]
    offsets: List[Tuple[int, int]] = enc["offset_mapping"]

    seg_ranges: List[Tuple[int, int, float, int, int]] = []
    cursor = 0
    for seg_idx, seg in enumerate(normalized_segments):
        seg_text = seg["text"]
        start = response_text.find(seg_text, cursor)
        if start == -1:
            start = cursor
        end = start + len(seg_text)
        seg_ranges.append((start, end, float(seg["value_score"]), seg_idx, int(seg["rank"])))
        cursor = end

    token_scores: List[float] = []
    token_segment_ids: List[int] = []
    token_segment_ranks: List[int] = []
    for tok_start, tok_end in offsets:
        midpoint = (tok_start + tok_end) / 2.0
        score = default_score
        seg_id = default_segment_id
        rank = default_rank
        for seg_start, seg_end, seg_score, seg_idx, seg_rank in seg_ranges:
            if seg_start <= midpoint < seg_end:
                score = seg_score
                seg_id = seg_idx
                rank = seg_rank
                break
        token_scores.append(score)
        token_segment_ids.append(seg_id)
        token_segment_ranks.append(rank)

    assert len(token_ids) == len(token_scores) == len(token_segment_ids) == len(token_segment_ranks), (
        "Mismatch while aligning token segment metadata"
    )
    return token_ids, token_scores, token_segment_ids, token_segment_ranks


# Quick sanity check
_test_segs = [
    {"text": "Hello world. ", "value_score": 0.8, "rank": 1},
    {"text": "How are you?", "value_score": 0.5, "rank": 2},
]
_test_resp = "".join(s["text"] for s in _test_segs)
_ids, _scores, _segment_ids, _segment_ranks = build_token_segment_metadata(
    _test_resp, _test_segs, tokenizer
)
print("Token IDs   :", _ids)
print("Token texts :", tokenizer.convert_ids_to_tokens(_ids))
print("Scores      :", _scores)
print("Segment ids :", _segment_ids)
print("Ranks       :", _segment_ranks)
assert len(_ids) == len(_scores) == len(_segment_ids) == len(_segment_ranks), "Length mismatch"
print("\nAlignment check passed.")


## Cell 5 - Preference Dataset

The loader is centered on the new combined schema. It skips rows that do not have both segmented responses because every requested method depends on segment scores and rankings. It also keeps a legacy path for older one-response-per-row files.


In [ ]:
class SegmentRankedPreferenceDataset(Dataset):
    """
    Loads the new combined JSONL format and returns preference triplets.

    Each item contains prompt ids, chosen/rejected response ids, and token-level
    segment score/id/rank metadata for both responses.
    """

    ROLE_MAP = {
        "chosen": "chosen",
        "positive": "chosen",
        "1": "chosen",
        "rejected": "rejected",
        "negative": "rejected",
        "0": "rejected",
    }

    def __init__(
        self,
        jsonl_path: str,
        tokenizer,
        max_length: int = 512,
    ):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples: List[Dict] = []
        self.skipped = {
            "missing_fields": 0,
            "missing_segments": 0,
            "alignment_or_truncation": 0,
            "legacy_incomplete_pairs": 0,
            "legacy_missing_role": 0,
        }
        self._load(jsonl_path)

    def _is_combined_record(self, rec: Dict) -> bool:
        return "positive_response" in rec or "negative_response" in rec

    def _load(self, path: str):
        legacy_raw: Dict[int, Dict[str, Dict]] = {}

        with open(path, encoding="utf-8") as f:
            for line_number, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue

                rec = json.loads(line)
                if self._is_combined_record(rec):
                    example = self._build_from_combined_record(rec)
                    if example is not None:
                        self.examples.append(example)
                    continue

                role = self._extract_role(rec)
                if role is None:
                    self.skipped["legacy_missing_role"] += 1
                    continue
                if not {"index", "prompt", "response", "segments"} <= rec.keys():
                    self.skipped["missing_fields"] += 1
                    continue
                legacy_raw.setdefault(int(rec["index"]), {})[role] = rec

        for idx in sorted(legacy_raw):
            pair = legacy_raw[idx]
            if "chosen" not in pair or "rejected" not in pair:
                self.skipped["legacy_incomplete_pairs"] += 1
                continue
            if pair["chosen"]["prompt"] != pair["rejected"]["prompt"]:
                raise ValueError(f"Prompt mismatch for legacy index {idx}")
            try:
                self.examples.append(self._build_example(pair["chosen"], pair["rejected"], idx=idx))
            except ValueError:
                self.skipped["alignment_or_truncation"] += 1

        print(f"Loaded {len(self.examples)} preference pairs from '{path}'.")
        print("Skipped rows:", self.skipped)

    def _build_from_combined_record(self, rec: Dict) -> Optional[Dict]:
        required = {"index", "prompt", "positive_response", "negative_response"}
        if not required <= rec.keys():
            self.skipped["missing_fields"] += 1
            return None
        if not rec.get("positive_segments") or not rec.get("negative_segments"):
            self.skipped["missing_segments"] += 1
            return None

        chosen_rec = {
            "prompt": rec["prompt"],
            "response": rec["positive_response"],
            "segments": rec["positive_segments"],
        }
        rejected_rec = {
            "prompt": rec["prompt"],
            "response": rec["negative_response"],
            "segments": rec["negative_segments"],
        }
        try:
            return self._build_example(chosen_rec, rejected_rec, idx=int(rec["index"]))
        except ValueError:
            self.skipped["alignment_or_truncation"] += 1
            return None

    @classmethod
    def _extract_role(cls, rec: Dict) -> Optional[str]:
        for key in ("role", "response_key", "label"):
            value = rec.get(key)
            if value is None:
                continue
            mapped = cls.ROLE_MAP.get(str(value).strip().lower())
            if mapped is not None:
                return mapped

        label_id = rec.get("label_id")
        if label_id is not None:
            mapped = cls.ROLE_MAP.get(str(label_id).strip())
            if mapped is not None:
                return mapped

        segs = rec.get("segments", [])
        if not segs:
            return None
        mean_score = sum(float(s.get("value_score", 0.0)) for s in segs) / len(segs)
        return "chosen" if mean_score >= 0.5 else "rejected"

    def _tokenize_prompt(self, prompt: str) -> List[int]:
        return self.tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_length // 2,
        )["input_ids"]

    def _build_response_tensors(self, response: str, segments: List[Dict], max_resp: int):
        token_ids, scores, segment_ids, segment_ranks = build_token_segment_metadata(
            response, segments, self.tokenizer
        )
        token_ids = token_ids[:max_resp]
        scores = scores[:max_resp]
        segment_ids = segment_ids[:max_resp]
        segment_ranks = segment_ranks[:max_resp]

        if not any(seg_id >= 0 for seg_id in segment_ids):
            raise ValueError("No aligned segment tokens remained after truncation")

        token_ids.append(self.tokenizer.eos_token_id)
        scores.append(0.0)
        segment_ids.append(-1)
        segment_ranks.append(-1)
        return token_ids, scores, segment_ids, segment_ranks

    def _build_example(self, chosen_rec: Dict, rejected_rec: Dict, idx: Optional[int] = None) -> Dict:
        prompt = chosen_rec["prompt"]
        prompt_ids = self._tokenize_prompt(prompt)
        max_resp = max(1, self.max_length - len(prompt_ids) - 1)

        chosen_ids, chosen_scores, chosen_segment_ids, chosen_segment_ranks = self._build_response_tensors(
            chosen_rec["response"], chosen_rec["segments"], max_resp
        )
        rejected_ids, rejected_scores, rejected_segment_ids, rejected_segment_ranks = self._build_response_tensors(
            rejected_rec["response"], rejected_rec["segments"], max_resp
        )

        return {
            "index": idx if idx is not None else -1,
            "prompt_ids": torch.tensor(prompt_ids, dtype=torch.long),
            "chosen_ids": torch.tensor(chosen_ids, dtype=torch.long),
            "chosen_segment_scores": torch.tensor(chosen_scores, dtype=torch.float),
            "chosen_segment_ids": torch.tensor(chosen_segment_ids, dtype=torch.long),
            "chosen_segment_ranks": torch.tensor(chosen_segment_ranks, dtype=torch.long),
            "rejected_ids": torch.tensor(rejected_ids, dtype=torch.long),
            "rejected_segment_scores": torch.tensor(rejected_scores, dtype=torch.float),
            "rejected_segment_ids": torch.tensor(rejected_segment_ids, dtype=torch.long),
            "rejected_segment_ranks": torch.tensor(rejected_segment_ranks, dtype=torch.long),
        }

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


# Increased from 384; 512 preserves headroom for two full models and four forwards.
DATASET_MAX_LENGTH = 512

dataset = SegmentRankedPreferenceDataset(str(DATA_PATH), tokenizer, max_length=DATASET_MAX_LENGTH)
if len(dataset) == 0:
    raise RuntimeError("No trainable examples were loaded. Check DATA_PATH and segment fields.")

sample = dataset[0]
print("\nSample keys:", list(sample.keys()))
print("Dataset max length:", DATASET_MAX_LENGTH)
print("Prompt tokens           :", sample["prompt_ids"].shape)
print("Chosen tokens           :", sample["chosen_ids"].shape)
print("Chosen segment scores   :", sample["chosen_segment_scores"][:20])
print("Chosen segment ids      :", sample["chosen_segment_ids"][:20])
print("Chosen segment ranks    :", sample["chosen_segment_ranks"][:20])
print("Rejected tokens         :", sample["rejected_ids"].shape)
print("Rejected segment scores :", sample["rejected_segment_scores"][:20])
print("Rejected segment ids    :", sample["rejected_segment_ids"][:20])
print("Rejected segment ranks  :", sample["rejected_segment_ranks"][:20])


## Cell 6 - Collate Function

The collator concatenates prompt and response tokens separately for chosen and rejected responses. Prompt, padding, unmatched whitespace, and EOS positions carry segment id/rank `-1`, so they are ignored by segment aggregation and structural PL terms.


In [ ]:
def dpo_collate_fn(batch: List[Dict], pad_token_id: int) -> Dict[str, torch.Tensor]:
    """Collate examples into padded tensors for chosen and rejected responses."""

    def pad_sequence_list(sequences: List[torch.Tensor], pad_val) -> torch.Tensor:
        max_len = max(s.size(0) for s in sequences)
        out = torch.full((len(sequences), max_len), pad_val, dtype=sequences[0].dtype)
        for i, s in enumerate(sequences):
            out[i, :s.size(0)] = s
        return out

    def build_full(prompt_ids, resp_ids, resp_segment_scores, resp_segment_ids, resp_segment_ranks):
        full_ids = torch.cat([prompt_ids, resp_ids])
        attn_mask = torch.ones(len(full_ids), dtype=torch.long)
        loss_mask = torch.cat([
            torch.zeros(len(prompt_ids), dtype=torch.long),
            torch.ones(len(resp_ids), dtype=torch.long),
        ])
        full_segment_scores = torch.cat([
            torch.zeros(len(prompt_ids), dtype=torch.float),
            resp_segment_scores,
        ])
        full_segment_ids = torch.cat([
            torch.full((len(prompt_ids),), -1, dtype=torch.long),
            resp_segment_ids,
        ])
        full_segment_ranks = torch.cat([
            torch.full((len(prompt_ids),), -1, dtype=torch.long),
            resp_segment_ranks,
        ])
        return full_ids, attn_mask, loss_mask, full_segment_scores, full_segment_ids, full_segment_ranks

    chosen_ids_list, chosen_attn_list, chosen_lmask_list = [], [], []
    chosen_score_list, chosen_segment_id_list, chosen_segment_rank_list = [], [], []
    rejected_ids_list, rejected_attn_list, rejected_lmask_list = [], [], []
    rejected_score_list, rejected_segment_id_list, rejected_segment_rank_list = [], [], []

    for ex in batch:
        c_ids, c_attn, c_lm, c_scores, c_seg_ids, c_seg_ranks = build_full(
            ex["prompt_ids"],
            ex["chosen_ids"],
            ex["chosen_segment_scores"],
            ex["chosen_segment_ids"],
            ex["chosen_segment_ranks"],
        )
        r_ids, r_attn, r_lm, r_scores, r_seg_ids, r_seg_ranks = build_full(
            ex["prompt_ids"],
            ex["rejected_ids"],
            ex["rejected_segment_scores"],
            ex["rejected_segment_ids"],
            ex["rejected_segment_ranks"],
        )
        chosen_ids_list.append(c_ids)
        chosen_attn_list.append(c_attn)
        chosen_lmask_list.append(c_lm)
        chosen_score_list.append(c_scores)
        chosen_segment_id_list.append(c_seg_ids)
        chosen_segment_rank_list.append(c_seg_ranks)
        rejected_ids_list.append(r_ids)
        rejected_attn_list.append(r_attn)
        rejected_lmask_list.append(r_lm)
        rejected_score_list.append(r_scores)
        rejected_segment_id_list.append(r_seg_ids)
        rejected_segment_rank_list.append(r_seg_ranks)

    return {
        "chosen_input_ids": pad_sequence_list(chosen_ids_list, pad_token_id),
        "chosen_attn_mask": pad_sequence_list(chosen_attn_list, 0),
        "chosen_loss_mask": pad_sequence_list(chosen_lmask_list, 0),
        "chosen_segment_scores": pad_sequence_list(chosen_score_list, 0.0),
        "chosen_segment_ids": pad_sequence_list(chosen_segment_id_list, -1),
        "chosen_segment_ranks": pad_sequence_list(chosen_segment_rank_list, -1),
        "rejected_input_ids": pad_sequence_list(rejected_ids_list, pad_token_id),
        "rejected_attn_mask": pad_sequence_list(rejected_attn_list, 0),
        "rejected_loss_mask": pad_sequence_list(rejected_lmask_list, 0),
        "rejected_segment_scores": pad_sequence_list(rejected_score_list, 0.0),
        "rejected_segment_ids": pad_sequence_list(rejected_segment_id_list, -1),
        "rejected_segment_ranks": pad_sequence_list(rejected_segment_rank_list, -1),
    }


from functools import partial

collate_fn = partial(dpo_collate_fn, pad_token_id=tokenizer.pad_token_id)

_loader = DataLoader(dataset, batch_size=2, collate_fn=collate_fn)
_batch = next(iter(_loader))
print("Batch keys:", list(_batch.keys()))
for k, v in _batch.items():
    print(f"  {k}: {tuple(v.shape)}  dtype={v.dtype}")


## Cell 7 - Segment Statistics and PL Helpers

This cell turns token log-probabilities into per-segment quantities needed by all five variants:

- `ell`: raw segment log-probability.
- `h`: policy-reference segment log-ratio.
- `h_bar`: per-token mean segment log-ratio.
- `U_theta`: Method A/C raw PL log-ratio between policy and reference ranking probabilities.
- `U_theta_omega`: Method A/C score-gap-weighted PL log-ratio.
- `logSC`: Method B structural coherence over importance-weighted segment log-ratios.


In [ ]:
def shifted_token_log_probs(
    logits: torch.Tensor,
    input_ids: torch.Tensor,
) -> torch.Tensor:
    """Return log p(token_t | prefix) for labels input_ids[:, 1:]."""
    shifted_logits = logits[:, :-1, :].contiguous()
    shifted_labels = input_ids[:, 1:].contiguous()
    # FP32 reduction prevents FP16 log-softmax underflow.
    log_probs = F.log_softmax(shifted_logits.float(), dim=-1)
    return log_probs.gather(
        dim=-1,
        index=shifted_labels.unsqueeze(-1),
    ).squeeze(-1)


def response_segment_statistics(
    logits: torch.Tensor,
    input_ids: torch.Tensor,
    loss_mask: torch.Tensor,
    segment_scores: torch.Tensor,
    segment_ids: torch.Tensor,
    segment_ranks: torch.Tensor,
) -> List[Dict[str, torch.Tensor]]:
    """Build per-sample, rank-ordered segment statistics from model logits."""
    token_log_probs = shifted_token_log_probs(logits, input_ids)
    shifted_mask = loss_mask[:, 1:].contiguous()
    shifted_scores = segment_scores[:, 1:].contiguous()
    shifted_segment_ids = segment_ids[:, 1:].contiguous()
    shifted_segment_ranks = segment_ranks[:, 1:].contiguous()

    batch_stats: List[Dict[str, torch.Tensor]] = []
    for b in range(token_log_probs.size(0)):
        valid = (shifted_mask[b] > 0) & (shifted_segment_ids[b] >= 0)
        if not torch.any(valid):
            empty = token_log_probs[b][valid]  # empty, but retains the autograd connection
            batch_stats.append({
                "ell": empty,
                "score": empty,
                "length": empty,
                "rank": torch.empty((0,), dtype=torch.long, device=token_log_probs.device),
            })
            continue

        sample_values = token_log_probs[b][valid]
        sample_scores = shifted_scores[b][valid]
        sample_segment_ids = shifted_segment_ids[b][valid]
        sample_segment_ranks = shifted_segment_ranks[b][valid]

        items = []
        for seg_id_tensor in torch.unique(sample_segment_ids, sorted=True):
            seg_id = int(seg_id_tensor.item())
            seg_mask = sample_segment_ids == seg_id
            ell = sample_values[seg_mask].sum()
            score = sample_scores[seg_mask][0].to(dtype=sample_values.dtype)
            length = seg_mask.sum().to(dtype=sample_values.dtype)
            rank = int(sample_segment_ranks[seg_mask][0].item())
            sort_rank = rank if rank >= 0 else 10**9
            items.append((sort_rank, seg_id, ell, score, length, rank))

        items.sort(key=lambda item: (item[0], item[1]))
        batch_stats.append({
            "ell": torch.stack([item[2] for item in items]),
            "score": torch.stack([item[3] for item in items]),
            "length": torch.stack([item[4] for item in items]),
            "rank": torch.tensor([item[5] for item in items], dtype=torch.long, device=token_log_probs.device),
        })

    return batch_stats


def pl_log_prob_ordered(utilities: torch.Tensor) -> torch.Tensor:
    """Plackett-Luce log probability for an already rank-ordered utility vector."""
    if utilities.numel() <= 1:
        return utilities.sum() * 0.0  # exact zero with a valid grad_fn
    terms = []
    for start in range(utilities.numel() - 1):
        terms.append(utilities[start] - torch.logsumexp(utilities[start:], dim=0))
    return torch.stack(terms).sum()


def pl_step_log_terms_ordered(utilities: torch.Tensor) -> torch.Tensor:
    """Per-step PL log terms for an already rank-ordered utility vector."""
    if utilities.numel() <= 1:
        return utilities.new_empty((0,))
    terms = []
    for start in range(utilities.numel() - 1):
        terms.append(utilities[start] - torch.logsumexp(utilities[start:], dim=0))
    return torch.stack(terms)


def omega_gap_weights_ordered(scores: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Score-aware omega weights from Eq. 32.

    scores are already ordered by rank. omega_t compares the current segment's
    score to the mean score of all lower-ranked remaining segments.
    """
    if scores.numel() <= 1:
        return scores.new_empty((0,))

    weights = []
    for start in range(scores.numel() - 1):
        lower_mean = scores[start + 1:].mean()
        weights.append((scores[start] - lower_mean).clamp_min(0.0))
    omega = torch.stack(weights)

    # If scores are tied or noisy enough to produce no positive gaps, fall back
    # to equal PL-step weights instead of dropping the structural signal.
    if omega.sum() <= eps:
        omega = torch.ones_like(omega)
    return omega


def omega_weighted_pl_log_ratio_ordered(
    policy_utilities: torch.Tensor,
    ref_utilities: torch.Tensor,
    scores: torch.Tensor,
    eps: float = 1e-8,
) -> torch.Tensor:
    """Normalized omega-weighted Method A/C structural utility."""
    policy_terms = pl_step_log_terms_ordered(policy_utilities)
    ref_terms = pl_step_log_terms_ordered(ref_utilities)
    if policy_terms.numel() == 0:
        return policy_utilities.sum() * 0.0  # exact zero with a valid grad_fn

    deltas = policy_terms - ref_terms
    omega = omega_gap_weights_ordered(scores.to(dtype=deltas.dtype), eps=eps)
    return (omega * deltas).sum() / omega.sum().clamp_min(eps)


def response_terms(
    policy_stats: Dict[str, torch.Tensor],
    ref_stats: Dict[str, torch.Tensor],
    beta: float,
    eps: float = 1e-8,
) -> Dict[str, torch.Tensor]:
    """Compute reusable response-level and structural terms."""
    if policy_stats["ell"].numel() == 0:
        zero = policy_stats["ell"].sum() * 0.0  # preserve policy autograd graph
        return {
            "standard_core": zero,
            "vdpo_core": zero,
            "method_a_utility": zero,
            "method_a_omega_utility": zero,
            "method_b_log_sc": zero,
        }

    ell_policy = policy_stats["ell"]
    ell_ref = ref_stats["ell"]
    scores = policy_stats["score"].to(dtype=ell_policy.dtype)
    lengths = policy_stats["length"].to(dtype=ell_policy.dtype).clamp_min(1.0)
    h = ell_policy - ell_ref

    standard_core = beta * h.sum()
    h_bar = h / lengths
    vdpo_h = (scores * h_bar).sum() / scores.sum().clamp_min(eps)
    vdpo_core = beta * vdpo_h

    method_a_utility = pl_log_prob_ordered(beta * ell_policy) - pl_log_prob_ordered(beta * ell_ref)
    method_a_omega_utility = omega_weighted_pl_log_ratio_ordered(
        beta * ell_policy,
        beta * ell_ref,
        scores,
        eps=eps,
    )
    method_b_log_sc = pl_log_prob_ordered(scores * beta * h)

    return {
        "standard_core": standard_core,
        "vdpo_core": vdpo_core,
        "method_a_utility": method_a_utility,
        "method_a_omega_utility": method_a_omega_utility,
        "method_b_log_sc": method_b_log_sc,
    }


print("Segment statistics and PL helpers defined successfully.")


## Cell 8 - Structured Preference Loss Module

The same loss class implements all requested variants through `METHOD_VARIANT`:

- `A`: Method A omega-weighted structural utility only.
- `B-DPO`: standard DPO core plus Method B log structural coherence.
- `B-VDPO`: VDPO core plus Method B log structural coherence.
- `C-DPO`: standard DPO core plus Method A omega-weighted structural utility.
- `C-VDPO`: VDPO core plus Method A omega-weighted structural utility.


In [ ]:
class StructuredPreferenceLoss(nn.Module):
    """Unified SRPO loss for Method A, B-DPO, B-VDPO, C-DPO, and C-VDPO."""

    SUPPORTED_VARIANTS = {"A", "B-DPO", "B-VDPO", "C-DPO", "C-VDPO"}

    def __init__(self, beta: float = 0.1, variant: str = METHOD_VARIANT, eps: float = 1e-8):
        super().__init__()
        if variant not in self.SUPPORTED_VARIANTS:
            raise ValueError(f"Unsupported method variant: {variant}")
        self.beta = beta
        self.variant = variant
        self.eps = eps

    def _objective_for_response(
        self,
        policy_stats: Dict[str, torch.Tensor],
        ref_stats: Dict[str, torch.Tensor],
    ) -> torch.Tensor:
        terms = response_terms(policy_stats, ref_stats, beta=self.beta, eps=self.eps)

        if self.variant == "A":
            return terms["method_a_omega_utility"]
        if self.variant == "B-DPO":
            return terms["standard_core"] + terms["method_b_log_sc"]
        if self.variant == "B-VDPO":
            return terms["vdpo_core"] + terms["method_b_log_sc"]
        if self.variant == "C-DPO":
            return terms["standard_core"] + terms["method_a_omega_utility"]
        if self.variant == "C-VDPO":
            return terms["vdpo_core"] + terms["method_a_omega_utility"]

        raise RuntimeError(f"Unhandled method variant: {self.variant}")

    def forward(
        self,
        policy_chosen_logits: torch.Tensor,
        policy_rejected_logits: torch.Tensor,
        ref_chosen_logits: torch.Tensor,
        ref_rejected_logits: torch.Tensor,
        chosen_input_ids: torch.Tensor,
        chosen_loss_mask: torch.Tensor,
        chosen_segment_scores: torch.Tensor,
        chosen_segment_ids: torch.Tensor,
        chosen_segment_ranks: torch.Tensor,
        rejected_input_ids: torch.Tensor,
        rejected_loss_mask: torch.Tensor,
        rejected_segment_scores: torch.Tensor,
        rejected_segment_ids: torch.Tensor,
        rejected_segment_ranks: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        policy_chosen_stats = response_segment_statistics(
            policy_chosen_logits,
            chosen_input_ids,
            chosen_loss_mask,
            chosen_segment_scores,
            chosen_segment_ids,
            chosen_segment_ranks,
        )
        policy_rejected_stats = response_segment_statistics(
            policy_rejected_logits,
            rejected_input_ids,
            rejected_loss_mask,
            rejected_segment_scores,
            rejected_segment_ids,
            rejected_segment_ranks,
        )

        with torch.no_grad():
            ref_chosen_stats = response_segment_statistics(
                ref_chosen_logits,
                chosen_input_ids,
                chosen_loss_mask,
                chosen_segment_scores,
                chosen_segment_ids,
                chosen_segment_ranks,
            )
            ref_rejected_stats = response_segment_statistics(
                ref_rejected_logits,
                rejected_input_ids,
                rejected_loss_mask,
                rejected_segment_scores,
                rejected_segment_ids,
                rejected_segment_ranks,
            )

        chosen_objectives = []
        rejected_objectives = []
        for b in range(len(policy_chosen_stats)):
            chosen_objectives.append(
                self._objective_for_response(policy_chosen_stats[b], ref_chosen_stats[b])
            )
            rejected_objectives.append(
                self._objective_for_response(policy_rejected_stats[b], ref_rejected_stats[b])
            )

        chosen_objectives = torch.stack(chosen_objectives)
        rejected_objectives = torch.stack(rejected_objectives)
        preference_logits = chosen_objectives - rejected_objectives
        loss = -F.logsigmoid(preference_logits).mean()
        preference_margin = preference_logits.mean().detach()

        return loss, chosen_objectives.detach(), rejected_objectives.detach(), preference_margin


preference_loss_fn = StructuredPreferenceLoss(beta=0.1, variant=METHOD_VARIANT)
print(
    "StructuredPreferenceLoss instantiated with "
    f"variant={preference_loss_fn.variant}, beta={preference_loss_fn.beta}"
)


## Cell 9 — Load Policy & Reference Models

In [ ]:
print(f"Loading full policy and frozen reference from '{MODEL_NAME}' ...")

# Policy: every parameter remains trainable. Checkpointed activations are the
# key activation-memory control for the four-forward custom preference loop.
policy_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    trust_remote_code=False,
    token=HF_TOKEN,
).to(DEVICE)
policy_model.config.use_cache = False
policy_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Reference must be a distinct frozen copy because full-model updates change the
# policy base weights. Loading from the cached checkpoint avoids a transient
# deepcopy and keeps the reference exactly at initialization.
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    trust_remote_code=False,
    token=HF_TOKEN,
).to(DEVICE)
ref_model.config.use_cache = False
ref_model.eval()
for parameter in ref_model.parameters():
    parameter.requires_grad_(False)

trainable_count = sum(p.numel() for p in policy_model.parameters() if p.requires_grad)
total_count = sum(p.numel() for p in policy_model.parameters())
assert trainable_count == total_count, "Full-model optimization requires every policy parameter"
print(f"Policy trainable parameters: {trainable_count:,} / {total_count:,} (100%)")
print("Gradient checkpointing:", policy_model.is_gradient_checkpointing)
print("Using dtype:", MODEL_DTYPE)
if DEVICE == "cuda":
    print(f"CUDA allocated after both models: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")


## Cell 10 — DataLoader & Optimizer

In [ ]:
# Full-model P100/16 GB defaults. Effective batch size = 1 * 8 = 8 pairs.
BATCH_SIZE         = 1
GRAD_ACCUM_STEPS   = 8
NUM_EPOCHS         = 1
LR                 = 2e-6
WARMUP_RATIO       = 0.05
GRAD_CLIP          = 1.0
SAVE_EVERY         = 1
LOG_EVERY          = 50        # optimizer steps
RUN_NAME           = "olmo2_0425_1b_sft_full_" + METHOD_VARIANT.lower().replace("-", "_")
OUTPUT_DIR         = Path("checkpoints") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=False,
    pin_memory=(DEVICE == "cuda"),
)

# PagedAdamW8bit still optimizes every full-model parameter with 8-bit moment
# states, but CUDA unified memory can page states to host RAM during VRAM peaks.
optimizer = bnb.optim.PagedAdamW8bit(
    policy_model.parameters(),
    lr=LR,
    weight_decay=0.01,
    percentile_clipping=100,
)

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = updates_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f"Method variant: {METHOD_VARIANT}")
print(f"Checkpoint directory: {OUTPUT_DIR}")
print(f"Sequence budget: {DATASET_MAX_LENGTH} tokens")
print(f"Batches per epoch: {len(train_loader)}")
print(f"Optimizer steps per epoch: {updates_per_epoch}")
print(f"Total optimizer steps: {total_steps}  |  Warmup steps: {warmup_steps}")
print(f"Effective batch size (pairs): {BATCH_SIZE * GRAD_ACCUM_STEPS}")


## Cell 11 — Training Loop

In [ ]:
from tqdm.auto import tqdm


def run_model(model, input_ids: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
    """Run a causal-LM forward pass and return logits."""
    output = model(
        input_ids=input_ids.to(DEVICE),
        attention_mask=attn_mask.to(DEVICE),
        use_cache=False,
    )
    return output.logits


def move_batch(batch: Dict, device: str) -> Dict:
    """Move all tensors in a batch dict to `device`."""
    return {k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()}


history = {"loss": [], "preference_margin": [], "step": []}
global_batch_step = 0
global_update_step = 0

policy_model.train()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0
    epoch_margin = 0.0
    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(
        train_loader,
        total=len(train_loader),
        desc=f"{METHOD_VARIANT} epoch {epoch}/{NUM_EPOCHS}",
        unit="batch",
        dynamic_ncols=True,
    )

    for batch_step, batch in enumerate(progress_bar, 1):
        global_batch_step += 1
        batch = move_batch(batch, DEVICE)

        policy_chosen_logits = run_model(
            policy_model,
            batch["chosen_input_ids"],
            batch["chosen_attn_mask"],
        )
        policy_rejected_logits = run_model(
            policy_model,
            batch["rejected_input_ids"],
            batch["rejected_attn_mask"],
        )

        with torch.no_grad():
            ref_chosen_logits = run_model(
                ref_model,
                batch["chosen_input_ids"],
                batch["chosen_attn_mask"],
            )
            ref_rejected_logits = run_model(
                ref_model,
                batch["rejected_input_ids"],
                batch["rejected_attn_mask"],
            )

        loss, chosen_obj, rejected_obj, preference_margin = preference_loss_fn(
            policy_chosen_logits=policy_chosen_logits,
            policy_rejected_logits=policy_rejected_logits,
            ref_chosen_logits=ref_chosen_logits,
            ref_rejected_logits=ref_rejected_logits,
            chosen_input_ids=batch["chosen_input_ids"],
            chosen_loss_mask=batch["chosen_loss_mask"],
            chosen_segment_scores=batch["chosen_segment_scores"],
            chosen_segment_ids=batch["chosen_segment_ids"],
            chosen_segment_ranks=batch["chosen_segment_ranks"],
            rejected_input_ids=batch["rejected_input_ids"],
            rejected_loss_mask=batch["rejected_loss_mask"],
            rejected_segment_scores=batch["rejected_segment_scores"],
            rejected_segment_ids=batch["rejected_segment_ids"],
            rejected_segment_ranks=batch["rejected_segment_ranks"],
        )

        raw_loss = loss.detach()
        raw_margin = preference_margin.detach()
        epoch_loss += raw_loss.item()
        epoch_margin += raw_margin.item()

        (loss / GRAD_ACCUM_STEPS).backward()

        # Drop large vocabulary logits and completed graphs before the next
        # forward; this avoids carrying roughly half a GiB of stale tensors.
        del (
            loss, chosen_obj, rejected_obj, preference_margin,
            policy_chosen_logits, policy_rejected_logits,
            ref_chosen_logits, ref_rejected_logits,
        )

        should_step = (
            batch_step % GRAD_ACCUM_STEPS == 0
            or batch_step == len(train_loader)
        )

        if should_step:
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            # Return unused cached blocks after each accumulated update.
            # This runs once per GRAD_ACCUM_STEPS, not once per micro-batch.
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            global_update_step += 1

            if global_update_step % LOG_EVERY == 0:
                avg_loss = epoch_loss / batch_step
                avg_margin = epoch_margin / batch_step
                print(
                    f"[{METHOD_VARIANT} epoch {epoch}/{NUM_EPOCHS}  "
                    f"Update {global_update_step}/{total_steps}]  "
                    f"loss={raw_loss.item():.4f}  "
                    f"avg_loss={avg_loss:.4f}  "
                    f"preference_margin={raw_margin.item():.4f}  "
                    f"lr={scheduler.get_last_lr()[0]:.2e}"
                )
                history["loss"].append(raw_loss.item())
                history["preference_margin"].append(raw_margin.item())
                history["step"].append(global_update_step)

        progress_bar.set_postfix(
            loss=f"{raw_loss.item():.4f}",
            margin=f"{raw_margin.item():.4f}",
            update=f"{global_update_step}/{total_steps}",
            refresh=False,
        )

    progress_bar.close()

    print(
        f"\n=== {METHOD_VARIANT} epoch {epoch} summary ===  "
        f"avg_loss={epoch_loss/len(train_loader):.4f}  "
        f"avg_margin={epoch_margin/len(train_loader):.4f}\n"
    )

    if epoch % SAVE_EVERY == 0:
        ckpt_path = OUTPUT_DIR / f"epoch_{epoch}"
        policy_model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        print(f"  Full-model checkpoint saved -> {ckpt_path}")

print("\nTraining complete.")


## Cell 12 — Training Curve Plot

In [ ]:
import matplotlib.pyplot as plt

if history["step"]:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history["step"], history["loss"], marker="o", color="steelblue")
    ax1.set_title(f"{METHOD_VARIANT} Loss")
    ax1.set_xlabel("Global step")
    ax1.set_ylabel("Loss")
    ax1.grid(True, alpha=0.3)

    ax2.plot(history["step"], history["preference_margin"], marker="s", color="darkorange")
    ax2.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax2.set_title("Preference Margin")
    ax2.set_xlabel("Global step")
    ax2.set_ylabel("Chosen objective - rejected objective")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"training_curves_{METHOD_VARIANT.lower().replace('-', '_')}.png", dpi=120)
    plt.show()
    print("Saved training curve image")
else:
    print("No log points yet (LOG_EVERY may be larger than total steps).")


## Cell 13 - Verification Tests


In [ ]:
print("=" * 60)
print(f"VERIFICATION TESTS: {METHOD_VARIANT}")
print("=" * 60)

# TEST 1: Dataset exposes segment ranks and skips unusable combined rows.
print("\n[TEST 1] Dataset schema and rank metadata")
assert len(dataset) > 0, "Dataset should contain trainable examples"
sample = dataset[0]
for key in [
    "chosen_segment_scores",
    "chosen_segment_ids",
    "chosen_segment_ranks",
    "rejected_segment_scores",
    "rejected_segment_ids",
    "rejected_segment_ranks",
]:
    assert key in sample, f"Missing {key}"
assert (sample["chosen_segment_ids"] >= 0).any(), "Chosen response has no aligned segment tokens"
assert (sample["rejected_segment_ids"] >= 0).any(), "Rejected response has no aligned segment tokens"
print(f"  Loaded trainable pairs: {len(dataset)}")
print("  Segment score/id/rank tensors are present.")

# TEST 2: Out-of-range scores are rank-derived instead of used directly.
print("\n[TEST 2] Out-of-range score repair")
dirty_segments = [
    {"text": "low ", "value_score": 3.0, "rank": 3},
    {"text": "high ", "value_score": 1.0, "rank": 1},
    {"text": "mid", "value_score": 2.0, "rank": 2},
]
clean_segments = normalize_segment_records(dirty_segments)
assert all(0.0 <= seg["value_score"] <= 1.0 for seg in clean_segments)
rank_to_score = {seg["rank"]: seg["value_score"] for seg in clean_segments}
assert rank_to_score[1] > rank_to_score[2] > rank_to_score[3]
print("  Rank-derived scores:", rank_to_score)

# TEST 3: PL utilities vanish for single-segment responses.
print("\n[TEST 3] Single-segment PL recovery")
one_utility = torch.tensor([0.5], dtype=torch.float32)
assert torch.allclose(pl_log_prob_ordered(one_utility), torch.tensor(0.0))
print("  PL log-prob is zero for one segment, as expected.")

# TEST 4: Response term functions are finite and ordered by rank.
print("\n[TEST 4] Response term finite values")
policy_stats = {
    "ell": torch.tensor([-1.0, -2.0, -3.0], dtype=torch.float32),
    "score": torch.tensor([0.9, 0.5, 0.1], dtype=torch.float32),
    "length": torch.tensor([2.0, 3.0, 4.0], dtype=torch.float32),
    "rank": torch.tensor([1, 2, 3], dtype=torch.long),
}
ref_stats = {
    "ell": torch.tensor([-1.2, -1.9, -2.8], dtype=torch.float32),
    "score": policy_stats["score"],
    "length": policy_stats["length"],
    "rank": policy_stats["rank"],
}
terms = response_terms(policy_stats, ref_stats, beta=preference_loss_fn.beta)
for key, value in terms.items():
    assert torch.isfinite(value), f"{key} is not finite"
assert terms["method_b_log_sc"] <= 1e-6, "PL log probability should be <= 0"
print({key: round(float(value), 6) for key, value in terms.items()})

# TEST 5: Omega weights emphasize larger score gaps and fall back on ties.
print("\n[TEST 5] Omega score-gap weights")
omega = omega_gap_weights_ordered(torch.tensor([0.9, 0.5, 0.1], dtype=torch.float32))
assert omega.numel() == 2
assert omega[0] > omega[1], "Top segment should get larger omega when its gap is larger"
tied_omega = omega_gap_weights_ordered(torch.tensor([0.4, 0.4, 0.4], dtype=torch.float32))
assert torch.allclose(tied_omega, torch.ones_like(tied_omega))
print("  Omega weights:", [round(float(x), 6) for x in omega])
print("  Tied-score fallback:", [round(float(x), 6) for x in tied_omega])

# TEST 6: Loss object is wired to this notebook's variant.
print("\n[TEST 6] Variant wiring")
assert preference_loss_fn.variant == METHOD_VARIANT
assert METHOD_VARIANT in StructuredPreferenceLoss.SUPPORTED_VARIANTS
print(f"  Active variant: {preference_loss_fn.variant}")

print("\nAll verification tests passed.")


## Cell 14 - Qualitative Generation Test
Prompt the trained model to inspect behavior after this method-specific run.


In [ ]:
policy_model.eval()


def _build_generation_inputs(prompt: str):
    if getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": prompt}]
        try:
            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        return tokenizer(formatted, return_tensors="pt").to(DEVICE)

    return tokenizer(prompt, return_tensors="pt").to(DEVICE)


def generate(model, prompt: str, max_new_tokens: int = 80) -> str:
    inputs = _build_generation_inputs(prompt)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return text.replace("<think>", "").replace("</think>", "").strip()


test_prompt = "Explain photosynthesis simply."
print(f"Prompt: {test_prompt}\n")
print("Policy model response:")
print(generate(policy_model, test_prompt))
print("\nReference model response:")
print(generate(ref_model, test_prompt))


## Summary

This notebook implements `C-DPO` from the SRPO method compendium.

| Component | Where |
|---|---|
| New combined dataset inspection and path resolution | Cell 3 |
| Token-to-segment alignment with score/rank repair | Cell 4 |
| Combined preference dataset loader | Cell 5 |
| Collation with segment score/id/rank tensors | Cell 6 |
| Segment statistics, VDPO aggregation, and PL helpers | Cell 7 |
| Unified structured preference loss | Cell 8 |
| Full OLMo-2-0425-1B-SFT policy and frozen reference loading | Cell 9 |
| Method-specific training loop and checkpointing | Cells 10-11 |
| Verification tests | Cell 13 |

Key design decisions:
- Rows without both `positive_segments` and `negative_segments` are skipped, because all requested losses require segment structure.
- Normal `value_score` fields in `[0, 1]` are preserved. Out-of-range score rows are repaired from `rank` so the intended ordering is not inverted.
- Method A/C use score-gap-weighted `U_theta_omega`, so high-score segment ordering mistakes matter more.
- Method B uses importance-weighted segment log-ratios in `logSC`, so segment scores affect the PL coherence utility directly.
- DPO variants use response-level summed segment log-ratios; VDPO variants use score-weighted mean segment log-ratios.

OLMo-2-0425-1B-SFT full-optimization notes:
- Checkpoints contain the full model, not an adapter.
- All policy parameters are trainable; only AdamW moment states are stored in 8-bit.
- P100 selects FP16 because Pascal has no native BF16; newer GPUs select BF16 automatically.
- If a fragmented Kaggle session OOMs, restart the kernel and reduce `DATASET_MAX_LENGTH`.
